# 00 - Diagnóstico y toma de decisiones sobre los datos

Antes de construir cualquier modelo, primero hay que entender con qué datos se cuenta:
qué hay en cada base, cuántos registros trae, si hay errores o datos raros, y si las
distintas fuentes se pueden juntar entre sí sin problemas.

Este notebook responde esas preguntas con código real sobre las 7 bases de datos
entregadas. Cada sección hace una pregunta puntual, la resuelve con código, y al final
deja escrita la **decisión** que se toma con esa evidencia 


In [1]:
import sys
sys.path.append('..')

import pandas as pd
from src.utils import cargar_tabla

pd.set_option('display.width', 120)
pd.options.display.float_format = '{:,.2f}'.format

## 1. ¿Qué bases de datos tenemos y qué hay en cada una?

Se entregaron 7 bases SQLite. Antes de analizar nada, hay que ver qué columnas trae cada
una, cuántos registros tiene y cómo se ven algunas filas de ejemplo.

In [2]:
bases = {
    'clientes':              ('../02_Datos/clientes/clientes.db', 'clientes'),
    'crean_aho_cte':          ('../02_Datos/crean_aho_cte/crean_aho_cte.db', 'crean_aho_cte'),
    'crean_bolsillos':        ('../02_Datos/crean_bolsillos/crean_bolsillos.db', 'crean_bolsillos'),
    'crean_fiducuenta':       ('../02_Datos/crean_fiducuenta/crean_fiducuenta.db', 'crean_fiducuenta'),
    'crean_inv_virtual_cdt':  ('../02_Datos/crean_inv_virtual_cdt/crean_inv_virtual_cdt.db', 'crean_inv_virtual_cdt'),
    'invesbot':               ('../02_Datos/invesbot/invesbot.db', 'invesbot'),
    'estimador_ing':          ('../02_Datos/estimador_ing/estimador_ing.db', 'estimador_ing'),
}

for nombre, (ruta, tabla) in bases.items():
    df = cargar_tabla(ruta, tabla)
    print(f'--- {nombre} ---')
    print('Filas:', len(df))
    print('Columnas:', list(df.columns))
    print()

--- clientes ---
Filas: 860231
Columnas: ['numero_id', 'grupo_edad', 'desc_genero', 'desc_segmento', 'desc_tipo_de_vivienda', 'ingresos_mensuales', 'total_egresos_mensuales', 'total_activos', 'total_pasivos', 'total_patrimonio']



--- crean_aho_cte ---
Filas: 1000000
Columnas: ['fecha', 'numero_id', 'producto', 'saldo']



--- crean_bolsillos ---
Filas: 1000000
Columnas: ['fecha', 'numero_id', 'producto', 'saldo']



--- crean_fiducuenta ---
Filas: 1000000
Columnas: ['fecha', 'numero_id', 'producto', 'saldo']



--- crean_inv_virtual_cdt ---
Filas: 994177
Columnas: ['fecha', 'numero_id', 'producto', 'saldo']



--- invesbot ---
Filas: 1000000
Columnas: ['fecha', 'numero_id', 'producto', 'saldo']



--- estimador_ing ---
Filas: 745792
Columnas: ['numero_id', 'producto', 'estimador_ingreso']



**Nota:** guardamos el diccionario `bases` porque lo vamos a volver a usar varias veces
en este notebook — así no repetimos las rutas de cada archivo cada vez.

In [3]:
# Carguemos clientes aparte, porque es la tabla principal (1 fila = 1 cliente)
clientes = cargar_tabla(*bases['clientes'])
clientes.head()

,numero_id,grupo_edad,desc_genero,desc_segmento,desc_tipo_de_vivienda,ingresos_mensuales,total_egresos_mensuales,total_activos,total_pasivos,total_patrimonio
0,8805210490649048784,65+,masculino,preferencial,NaN,"29,239,444.00","30,000,000.00","145,047,043,000.00","6,356,702,000.00","105,422,323.00"
1,-5723572980375902369,50-65,masculino,preferencial,PROPIA,"31,024,544.00","10,000,000.00","1,133,345,000.00","83,072,000.00","1,050,273,000.00"
2,-8245474570363424359,65+,masculino,preferencial,PROPIA,"2,834,000.00","500,000.00","1,073,787,017.00",0.00,"343,000,000.00"
3,-7840506796880772723,65+,femenino,preferencial,PROPIA,"28,035,850.00",0.00,"175,000,000.00",0.00,"55,000,000.00"
4,5309731180094827430,36-49,femenino,preferencial,FAMILIAR,"3,846,205.00","2,500,000.00","758,000,000.00",0.00,"758,000,000.00"


In [4]:
clientes.dtypes

numero_id                    int64
grupo_edad                     str
desc_genero                    str
desc_segmento                  str
desc_tipo_de_vivienda          str
ingresos_mensuales         float64
total_egresos_mensuales    float64
total_activos              float64
total_pasivos              float64
total_patrimonio           float64
dtype: object

## 2. ¿Hay clientes repetidos?

Si el mismo número de identificación aparece dos veces en la tabla de clientes, ese
cliente contaría doble en cualquier cálculo (por ejemplo, si es candidato para la App,
aparecería dos veces en la lista de priorización). Por eso hay que revisarlo antes de
seguir.

In [5]:
duplicados = clientes['numero_id'].duplicated().sum()
print('Clientes con numero_id repetido:', duplicados, 'de', len(clientes))

Clientes con numero_id repetido: 8 de 860231


**Decisión:** si el número de duplicados es pequeño frente al total de clientes, en el
notebook de limpieza nos quedamos solo con la primera fila de cada cliente repetido y
descartamos la(s) demás.

## 3. ¿Hay datos vacíos (nulos)?

Un dato vacío no es necesariamente un error — puede significar que esa información nunca
se capturó para ese cliente. Pero hay que saber cuántos hay y en qué columnas, para
decidir qué hacer con cada caso (no es lo mismo que falte en el 1% de los casos que en el
70%).

In [6]:
nulos = clientes.isnull().sum()
porcentaje_nulos = (nulos / len(clientes) * 100).round(1)
resumen_nulos = pd.DataFrame({'nulos': nulos, 'porcentaje_%': porcentaje_nulos})
resumen_nulos

,nulos,porcentaje_%
numero_id,0,0.00
grupo_edad,0,0.00
desc_genero,93,0.00
desc_segmento,0,0.00
desc_tipo_de_vivienda,591699,68.80
ingresos_mensuales,249,0.00
total_egresos_mensuales,249,0.00
total_activos,249,0.00
total_pasivos,249,0.00
total_patrimonio,260,0.00


**Decisión:** para columnas con un porcentaje de vacíos muy alto, no tiene sentido
"adivinar" el valor — mejor crear una categoría propia tipo "no informa". Para columnas
con muy pocos vacíos, se puede rellenar con un valor típico (la mediana) o simplemente
excluir esas pocas filas, porque el impacto es mínimo. La regla exacta para cada columna
se define en el notebook de limpieza, con base en el porcentaje que salga aquí.

## 4. ¿Hay valores que no tienen sentido? (valores extremos)

Ahora miramos si hay números financieros fuera de lo razonable — por ejemplo, ingresos
mensuales absurdamente altos, que casi seguro son errores de digitación y no clientes
reales.

In [7]:
clientes[['ingresos_mensuales', 'total_egresos_mensuales', 'total_activos',
          'total_pasivos', 'total_patrimonio']].describe()

,ingresos_mensuales,total_egresos_mensuales,total_activos,total_pasivos,total_patrimonio
count,"859,982.00","859,982.00","859,982.00","859,982.00","859,971.00"
mean,"39,323,048.76","147,800,881.98","199,790,079.43","96,876,727.99","131,115,604.88"
std,"16,996,731,708.31","26,467,126,565.12","21,327,395,630.97","22,461,067,642.83","26,894,670,043.30"
min,0.00,0.00,0.00,0.00,"-8,999,512,000,000.00"
25%,"1,300,000.00","100,000.00","1,145,000.00",0.00,"153,615.00"
50%,"2,000,000.00","400,000.00","10,902,000.00",0.00,"7,000,000.00"
75%,"4,889,725.75","800,000.00","66,935,362.25",1.00,"47,346,500.00"
max,"9,000,008,796,700.00","8,500,000,850,000.00","9,041,914,166,666.00","9,000,012,000,000.00","9,041,902,166,666.00"


In [8]:
# Miremos puntualmente cuántos clientes tienen un ingreso mensual "imposible"
# (por encima de un límite que ya no tiene sentido para una persona natural)
umbral = 1_000_000_000  # mil millones de pesos al mes
casos_extremos = clientes[clientes['ingresos_mensuales'] > umbral]
print('Clientes con ingresos mensuales reportados por encima de', umbral, ':', len(casos_extremos))
casos_extremos[['numero_id', 'ingresos_mensuales', 'total_patrimonio']].head(10)

Clientes con ingresos mensuales reportados por encima de 1000000000 : 56


,numero_id,ingresos_mensuales,total_patrimonio
44700,-370971257901166478,"19,446,813,000.00","733,842,000,000.00"
46104,899782306101282361,"1,038,333,438.00","78,475,000.00"
48463,1168418875456587865,"5,850,000,000.00","123,170,000.00"
68759,-3085604672515458652,"9,000,003,000,000.00","37,872,715.00"
100601,8730911288060383845,"8,500,000,000.00","45,000,000,000.00"
100896,-4924496316621166591,"3,200,000,000.00","9,000,000.00"
124524,1140305611412498623,"3,228,517,240.00","22,840,000.00"
158700,6258068894549991637,"9,000,008,796,700.00","248,083,000.00"
173548,-510692888618568015,"2,748,191,000.00","14,352,966,000.00"
187430,-833163035508958536,"3,924,608,031.00","5,989,442,972.00"


**Decisión:** si estos casos son pocos frente al total de clientes, se excluyen o se
"recortan" a un techo razonable antes de entrenar cualquier modelo — dejarlos tal cual
puede dañar promedios y coeficientes de todo el análisis, aunque sean solo un puñado de
filas.

## 5. ¿Los identificadores de cliente coinciden entre todas las tablas?

Cada producto (ahorro, bolsillos, fiducuenta, etc.) está en una tabla separada. Antes de
poder juntar todo en una sola tabla de cliente, hay que confirmar que el número de
identificación que usan es el mismo, y que no hay identificaciones "huérfanas" — es decir,
que aparecen en la tabla de un producto pero no existen en la tabla de clientes.

In [9]:
ids_clientes = set(clientes['numero_id'])

for nombre, (ruta, tabla) in bases.items():
    if nombre == 'clientes':
        continue
    df = cargar_tabla(ruta, tabla)
    ids_tabla = set(df['numero_id'].unique())
    fuera_de_clientes = ids_tabla - ids_clientes
    porcentaje_ok = 100 * (len(ids_tabla) - len(fuera_de_clientes)) / len(ids_tabla)
    print(f'{nombre}: {len(ids_tabla)} clientes distintos | '
          f'{len(fuera_de_clientes)} no existen en "clientes" | '
          f'{porcentaje_ok:.1f}% consistente')

crean_aho_cte: 475719 clientes distintos | 0 no existen en "clientes" | 100.0% consistente


crean_bolsillos: 260714 clientes distintos | 0 no existen en "clientes" | 100.0% consistente


crean_fiducuenta: 181021 clientes distintos | 0 no existen en "clientes" | 100.0% consistente


crean_inv_virtual_cdt: 84104 clientes distintos | 0 no existen en "clientes" | 100.0% consistente


invesbot: 5214 clientes distintos | 0 no existen en "clientes" | 100.0% consistente


estimador_ing: 745792 clientes distintos | 0 no existen en "clientes" | 100.0% consistente


**Decisión:** si todas las tablas salen en (o muy cerca de) 100% consistentes, se puede
usar `numero_id` como llave para juntar las tablas sin necesidad de mapeos ni correcciones
adicionales.

## 6. ¿Con qué frecuencia se registró cada producto en el tiempo?

Cada tabla de producto trae varias fotos del saldo del cliente en distintas fechas. La
pregunta es si todas las tablas se midieron con la misma frecuencia (por ejemplo, una vez
al mes) o si unas se midieron más seguido que otras.

In [10]:
for nombre, (ruta, tabla) in bases.items():
    if nombre in ('clientes', 'estimador_ing'):
        continue  # estas dos no tienen columna de fecha
    df = cargar_tabla(ruta, tabla)
    fechas_distintas = df['fecha'].nunique()
    registros_por_cliente = df.groupby('numero_id').size()
    print(f'{nombre}: {fechas_distintas} fechas distintas | '
          f'en promedio {registros_por_cliente.mean():.1f} registros por cliente '
          f'(min {registros_por_cliente.min()}, max {registros_por_cliente.max()})')

crean_aho_cte: 91 fechas distintas | en promedio 2.1 registros por cliente (min 1, max 12)


crean_bolsillos: 37 fechas distintas | en promedio 3.8 registros por cliente (min 1, max 13)


crean_fiducuenta: 64 fechas distintas | en promedio 5.5 registros por cliente (min 1, max 12)


crean_inv_virtual_cdt: 391 fechas distintas | en promedio 11.8 registros por cliente (min 1, max 199)


invesbot: 389 fechas distintas | en promedio 191.8 registros por cliente (min 1, max 340)


**Decisión:** si la frecuencia no es igual entre tablas, no conviene sumar o promediar
todos los registros tal cual, porque los productos medidos más seguido pesarían más sin
razón de negocio. En vez de eso, para cada cliente y cada producto se toma solo el
**saldo más reciente conocido** — una sola foto por producto, medida con la misma vara
para todos.

## 7. ¿El ingreso declarado coincide con el ingreso estimado por comportamiento?

Hay dos fuentes de ingreso distintas: lo que el cliente declaró (`ingresos_mensuales`, en
la tabla `clientes`) y lo que el banco calculó mirando su comportamiento transaccional
(`estimador_ingreso`, en la tabla `estimador_ing`). Si las dos cuentan una historia
parecida, es una buena señal de que los datos son confiables.

In [11]:
estimador = cargar_tabla(*bases['estimador_ing'])
cruce = clientes[['numero_id', 'ingresos_mensuales']].merge(
    estimador[['numero_id', 'estimador_ingreso']], on='numero_id', how='inner'
)
print('Clientes con las dos fuentes de ingreso:', len(cruce))
print('Correlación (con outliers incluidos):', cruce['ingresos_mensuales'].corr(cruce['estimador_ingreso']))

Clientes con las dos fuentes de ingreso: 745792
Correlación (con outliers incluidos): 0.0005518902987009941


In [12]:
# Repetimos la correlación pero quitando los ingresos "imposibles" que ya identificamos en la sección 4
cruce_filtrado = cruce[cruce['ingresos_mensuales'] < umbral]
print('Clientes despues de quitar outliers:', len(cruce_filtrado))
print('Correlación (sin outliers):', cruce_filtrado['ingresos_mensuales'].corr(cruce_filtrado['estimador_ingreso']))

Clientes despues de quitar outliers: 745568
Correlación (sin outliers): 0.6271638440269899


**Decisión:** si la correlación mejora mucho al quitar los outliers, eso confirma que el
problema real está en esos pocos valores extremos (no en que las fuentes sean
inconsistentes) — y que `estimador_ingreso` es una fuente confiable para completar casos
donde falte `ingresos_mensuales`.

## 8. ¿Quién ya usa un producto de inversión hoy?

`Invesbot` es el producto más parecido a la nueva App (es un servicio digital de inversión
con recomendaciones automáticas). Antes de usarlo como referencia para el modelo, hay que
confirmar que su perfil de adopción tiene sentido de negocio: ¿los clientes que ya lo
tienen se ven distintos del resto?

In [13]:
invesbot_df = cargar_tabla(*bases['invesbot'])
ids_invesbot = set(invesbot_df['numero_id'].unique())

print('Clientes con Invesbot:', len(ids_invesbot), 'de', len(clientes),
      f'({100*len(ids_invesbot)/len(clientes):.2f}%)')

Clientes con Invesbot: 5214 de 860231 (0.61%)


In [14]:
clientes_temp = clientes.copy()
clientes_temp['tiene_invesbot'] = clientes_temp['numero_id'].isin(ids_invesbot)

clientes_temp.groupby('tiene_invesbot')[
    ['ingresos_mensuales', 'total_patrimonio']
].median()

,ingresos_mensuales,total_patrimonio
tiene_invesbot,,
False,"2,000,000.00","6,910,000.00"
True,"6,528,356.50","65,278,350.00"


In [15]:
# ¿La tasa de adopcion de Invesbot cambia segun el segmento comercial del cliente?
(pd.crosstab(clientes_temp['desc_segmento'], clientes_temp['tiene_invesbot'], normalize='index') * 100).round(2)

tiene_invesbot,False,True
desc_segmento,,
personal,99.82,0.18
plus,98.39,1.61
preferencial,95.13,4.87


**Decisión:** si se ve una diferencia clara de ingreso/patrimonio/segmento entre quienes
tienen Invesbot y quienes no, eso confirma que hay una señal real de negocio detrás — no
es ruido — y respalda usar "tiene Invesbot" como variable objetivo (aproximación) para el
modelo de propensión de adopción de la nueva App.

## 9. ¿Cuántos clientes ya invierten, cuántos son solo transaccionales y cuántos no
aparecen en ninguna tabla de producto?

Esto nos da una primera foto del tamaño de cada grupo, útil tanto para el modelo como para
el dimensionamiento final del negocio.

In [16]:
aho = set(cargar_tabla(*bases['crean_aho_cte'])['numero_id'].unique())
bol = set(cargar_tabla(*bases['crean_bolsillos'])['numero_id'].unique())
fid = set(cargar_tabla(*bases['crean_fiducuenta'])['numero_id'].unique())
cdt = set(cargar_tabla(*bases['crean_inv_virtual_cdt'])['numero_id'].unique())

transaccional = aho | bol
inversion = fid | cdt | ids_invesbot

ya_invierte = inversion
solo_transaccional = transaccional - inversion
sin_ninguna_senal = ids_clientes - transaccional - inversion

print(f'Ya tiene algun producto de inversion: {len(ya_invierte)} ({100*len(ya_invierte)/len(ids_clientes):.1f}%)')
print(f'Solo transaccional, sin inversion:    {len(solo_transaccional)} ({100*len(solo_transaccional)/len(ids_clientes):.1f}%)')
print(f'Sin ninguna senal en la muestra:      {len(sin_ninguna_senal)} ({100*len(sin_ninguna_senal)/len(ids_clientes):.1f}%)')

Ya tiene algun producto de inversion: 220452 (25.6%)
Solo transaccional, sin inversion:    310188 (36.1%)
Sin ninguna senal en la muestra:      329583 (38.3%)


**Decisión:** el grupo "ya invierte" son los candidatos más obvios (cross-sell). El grupo
"solo transaccional" es la oportunidad de activación más grande. El grupo "sin ninguna
señal" se incluye igual en el modelo (tiene datos demográficos completos en `clientes`),
pero se documenta como supuesto: no podemos confirmar si realmente no tienen el producto o
si simplemente no salieron en la muestra que nos entregaron (recordemos que cada tabla
viene topada en 1 millón de filas).

## 10. Resumen — reglas de limpieza que se aplican en el notebook `01_integracion_y_limpieza`

Con base en todo lo anterior, estas son las reglas que se van a aplicar:

1. **Clientes repetidos:** quedarnos con una sola fila por `numero_id`.
2. **`desc_tipo_de_vivienda` (muchos vacíos):** crear categoría `"no informa"` en vez de
   adivinar.
3. **Vacíos pequeños en variables financieras:** rellenar con la mediana o excluir esas
   pocas filas.
4. **Ingresos/patrimonio con valores imposibles:** excluir o recortar a un techo razonable
   antes de modelar.
5. **Texto con errores de codificación** (ej. `INVERSI�N VIRTUAL`): corregir el texto para
   que se lea bien.
6. **Frecuencia de medición distinta entre productos:** usar el saldo más reciente
   conocido por cliente y producto, no todo el historial.
7. **Clientes sin señal en alguna tabla de producto:** tratarlos como "sin evidencia del
   producto", sin excluirlos del análisis.
8. **`numero_id` como llave de integración:** ya validado que es consistente entre todas
   las tablas, se usa directamente para juntar todo en la tabla `Cliente 360`.